In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_excel("finalized datasets/vessel_dataset.xlsx")
# Check basic info
df.shape, df.columns

((850, 33),
 Index(['vessel_id', 'vessel_name', 'ship_type', 'year_built', 'loa_m', 'lpp_m',
        'breadth_m', 'depth_m', 'draft_m', 'displacement_t', 'dwt_t',
        'block_coefficient', 'service_speed_kn', 'payload_t', 'engine_type',
        'installed_power_kw', 'steel_weight_t', 'teu_capacity', 'level1_group',
        'size_band', 'strata', 'dataset_source', 'bollard_pull_t',
        'propulsion_type', 'number_of_engines', 'propeller_diameter_m',
        'fuel_capacity_m3', 'deck_area_m2', 'deck_load_t',
        'fresh_water_capacity_m3', 'cargo_pump_capacity_m3hr',
        'number_of_thrusters', 'dp_class'],
       dtype='object'))

In [4]:
# ---- Feature Engineering ---- #

# Ratios
df["L_B"] = df["loa_m"] / df["breadth_m"]
df["B_D"] = df["breadth_m"] / df["depth_m"]
df["L_D"] = df["loa_m"] / df["depth_m"]

# Speed-Length Ratio (using service_speed_kn)
df["speed_length_ratio"] = df["service_speed_kn"] / np.sqrt(df["loa_m"])

# Year Bucket
df["year_bucket"] = pd.cut(
    df["year_built"],
    bins=[0, 2000, 2010, 2020, 2030],
    labels=["old", "mid", "modern", "latest"]
)

# Drop rows where target is missing
df = df.dropna(subset=["installed_power_kw", "steel_weight_t"])

# Check new columns
df[["L_B", "B_D", "L_D", "speed_length_ratio", "year_bucket"]].head()

,L_B,B_D,L_D,speed_length_ratio,year_bucket
0,6.000000,1.875000,11.250000,1.043498,mid
1,6.250000,1.882353,11.764706,1.060660,mid
2,6.363636,1.833333,11.666667,1.035098,mid
3,6.470588,1.837838,11.891892,1.078720,mid
4,6.571429,1.842105,12.105263,1.055009,modern


In [5]:
# ---- Define Targets ---- #

y_power = df["installed_power_kw"]
y_weight = df["steel_weight_t"]

# ---- Drop unnecessary columns ---- #

drop_cols = [
    "installed_power_kw",
    "steel_weight_t",
    "vessel_id",
    "vessel_name"
]

X = df.drop(columns=drop_cols, errors="ignore")

X.shape

(850, 34)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Automatically detect categorical + numeric columns

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numeric:", len(numeric_cols))

# ---- Preprocessing ---- #

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

# ---- Model ---- #

model = RandomForestRegressor(random_state=42)

# ---- Pipeline ---- #

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

Categorical: ['ship_type', 'engine_type', 'level1_group', 'size_band', 'strata', 'dataset_source', 'propulsion_type', 'dp_class', 'year_bucket']
Numeric: 25


In [10]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV

# ---- Train-Test Split ---- #
X_train, X_test, y_train, y_test = train_test_split(
    X, y_power, test_size=0.2, random_state=42
)

# ---- Hyperparameter Grid ---- #
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

# ---- Random Search ---- #
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=5,   # keep small for speed
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

# ---- Train ---- #
search.fit(X_train, y_train)

best_pipeline_power = search.best_estimator_

print("Best Params:", search.best_params_)

Best Params: {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_depth': None}


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predictions
y_pred = best_pipeline_power.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 1376.2323529411765
RMSE: 3774.593134748978
R2 Score: 0.9771456981375217


In [12]:
# ---- Train-Test Split for Weight ---- #
X_train, X_test, y_train_w, y_test_w = train_test_split(
    X, y_weight, test_size=0.2, random_state=42
)

# ---- Train ---- #
search.fit(X_train, y_train_w)

best_pipeline_weight = search.best_estimator_

print("Weight model trained ✅")

Weight model trained ✅


In [13]:
# Predictions
y_pred_w = best_pipeline_weight.predict(X_test)

# Metrics
mae_w = mean_absolute_error(y_test_w, y_pred_w)
rmse_w = np.sqrt(mean_squared_error(y_test_w, y_pred_w))
r2_w = r2_score(y_test_w, y_pred_w)

print("MAE (Weight):", mae_w)
print("RMSE (Weight):", rmse_w)
print("R2 Score (Weight):", r2_w)

MAE (Weight): 311.3564413283306
RMSE (Weight): 1298.4968292196938
R2 Score (Weight): 0.9809289950024489


In [14]:
import joblib

# Save models
joblib.dump(best_pipeline_power, "power_pipeline.pkl")
joblib.dump(best_pipeline_weight, "weight_pipeline.pkl")

print("Models saved successfully ✅")

Models saved successfully ✅
